# 00. Что предсказывает DiT: path, velocity и направление времени

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf).
<!-- study-navigation:end -->

Отделим encoder/decoder, neural field, solver и policy. Выведем все соглашения до первого transformer forward.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lecture 1, notes §2.1 pp.7–9; Lecture 2, notes §3 pp.14–24, Lab 2 Q2.1–Q2.3/Q3.1. Сначала выведите probability path и conditional target; затем измените направление времени.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='00_probability_path_and_learning_route';metrics={}
print('Pinned source files checked:',check_sources())


## 1. От распределения к траектории

MIT часто ведёт время s от шума к данным: y_s=(1−s)ε+s z. В этих K6 inference helpers удобна противоположная координата σ: чистые данные при 0, шум при 1.

$$
x_\sigma=(1-\sigma)z+\sigma\epsilon,\qquad
u=\frac{d x_\sigma}{d\sigma}=\epsilon-z,\qquad s=1-\sigma.
$$

Следовательно dx/ds=−u. Одинаковый tensor нельзя подставлять как velocity после смены time convention без знака. Исходный conditional target доступен при обучении, когда известны z и ε; на inference чистый z неизвестен.

$$
p_\sigma(x)=\int p_\sigma(x\mid z)p_{\mathrm{data}}(z)\,dz,\qquad
u_\star(x,\sigma,c)=\mathbb{E}[\epsilon-z\mid x_\sigma=x,c].
$$

Условное ожидание возникает как minimizer squared regression: разложим E||f−U||² на E||f−E[U|X]||² и conditional variance, не зависящую от f. Отсюда смысл FM training, а не «нейросеть просто угадывает noise».


In [ ]:
z=torch.tensor([[.3,-.2]],dtype=torch.float64);noise=torch.tensor([[1.1,.7]],dtype=torch.float64)
sigma=.6;u=noise-z;x=(1-sigma)*z+sigma*noise
assert torch.allclose(x-sigma*u,z)
h=1e-5
fd=(((1-sigma-h)*z+(sigma+h)*noise)-((1-sigma+h)*z+(sigma-h)*noise))/(2*h)
assert torch.allclose(fd,u,atol=1e-10)
metrics['affine_velocity_fd_max_error']=float((fd-u).abs().max())
metrics['clean_endpoint_reparameterization_error']=float((x-sigma*u-z).abs().max())
print('x_sigma:',x,'u_sigma:',u,'u_s:',-u)


### Мост к MIT Lecture 3A: score не равен velocity

Для xσ=(1−σ)z+σε conditional covariance равна σ²I. Дифференцирование log Gaussian по x и posterior averaging дают:

$$
\nabla_x\log p_\sigma(x)=-\frac{x-(1-\sigma)\mathbb{E}[z\mid x]}{\sigma^2}.
$$

Подставим E[z|x]=x−σu⋆ и сократим σ:

$$
\nabla_x\log p_\sigma(x)=-\frac{x+(1-\sigma)u_\star(x,\sigma)}{\sigma},\qquad 0<\sigma<1.
$$

Это вывод для выбранного affine path, не универсальное тождество всех diffusion parameterizations. Проверим многомерную Gaussian с коррелированной covariance: z∼N(μ,Σ), Cσ=(1−σ)²Σ+σ²I. По Gaussian conditioning:

$$
\mathbb{E}[z\mid x]=\mu+(1-\sigma)\Sigma C_\sigma^{-1}(x-(1-\sigma)\mu).
$$

У density score напрямую равен −Cσ⁻¹(x−(1−σ)μ). Сравнение двух путей проверяет одновременно marginalization, covariance и velocity→score conversion. При σ=0 формулы с division нельзя подставлять без отдельного limit analysis.


In [ ]:
mu=torch.tensor([.2,-.3],dtype=torch.float64)
Sigma=torch.tensor([[1.,.4],[.4,2.]],dtype=torch.float64);sig=.4
cov=(1-sig)**2*Sigma+sig**2*torch.eye(2,dtype=torch.float64)
x=torch.randn(16,2,dtype=torch.float64);delta=x-(1-sig)*mu
post_mean=mu+delta@torch.linalg.solve(cov,(1-sig)*Sigma).T
u_marginal=(x-post_mean)/sig
score_from_velocity=-(x+(1-sig)*u_marginal)/sig
score_direct=-torch.linalg.solve(cov,delta.T).T
assert torch.allclose(score_from_velocity,score_direct,atol=1e-12)
metrics['correlated_Gaussian_score_bridge_max_error']=float((score_from_velocity-score_direct).abs().max())


## 2. Почему solver не равен DiT

Neural field Gθ выдаёт velocity той же **физической** размерности, что latent. Euler использует его как правую часть ODE:

$$
x_{k+1}=x_k+(\sigma_{k+1}-\sigma_k)G_\theta(x_k,1000\sigma_k,c).
$$

При denoising Δσ<0. Множитель 1000 — conditioning convention сети; Euler всё равно шагает по σ, а не на 1000Δσ. VAE только задаёт пространство z. Он не выполняет эту ODE.

В π-Flow student вместо одного velocity выдаёт параметры π. Между двумя student вызовами solver многократно вычисляет **обычную tensor function policy**, а не transformer. Поэтому отдельно считаем network NFE, grid points G и policy evaluations.


In [ ]:
x=noise.clone()
for sigma_now,sigma_next in zip(torch.linspace(1,0,9)[:-1],torch.linspace(1,0,9)[1:]):
    x=x+(sigma_next-sigma_now)*u
assert torch.allclose(x,z,atol=1e-7)
metrics['constant_field_euler_error']=float((x-z).abs().max())
# A dimension/units ledger, not real inference.
metrics['contracts']={'video_latent':'B,T,H,W,Cv','audio_latent':'B,A,Ca',
 'standard_head':'same physical C','grid_head':'B,G,...,C; state C unchanged',
 'solver_time':'sigma in [0,1]','network_time':'1000*sigma'}
print(metrics['contracts'])


## 3. Порядок нового маршрута

00 probability/ODE → 01 DiT primitives → 02 one generated visual stream + text CrossDiT → 03 Audio/Video fusion → 04 semantic/codec conditioning → 05 visual references → 06 Euler/CFG loop → 07 DX mathematics → 08 native π sampler → 09 miniature imitation distillation → 10 ComfyUI/manual experiment.

Существующие K6 labs 05–12 дают overview; здесь каждый из запрошенных файлов получает отдельный подробный разбор. Для codec вернитесь в [vae](../../vae/README.md). [MIT_BRIDGE](../MIT_BRIDGE.md) указывает исходные вопросы и границу дополнений.


## Измерение → вывод

Алгебра affine path и знак подтверждены численно. Constant-field Euler exact не доказывает exactness для нелинейного trained field; следующий шаг — понять параметризацию Gθ.


## Самостоятельные опыты

1. Выведите target при s=1−σ, затем проверьте finite difference.
2. Для xσ−σu объясните, когда это истинный z, а когда лишь predicted posterior mean.
3. Запишите shape ledger для clip и audio разной длины.

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
